In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec
import numpy.ma as ma

In [8]:
def compute_ij(step_size, mask_size):
    # Generate positions for updating symmetry image
    positions = []
    for i in range(mask_size):
        for j in range(0, mask_size, step_size):
            positions.append([i, j])
        positions.append([i, mask_size - 1])
    return np.array(positions)

def update_mask(row_ind, col_ind, step_size, mask_size):
    mask = np.ones((mask_size, mask_size), dtype=bool)  # Masking all
    i = max(0, row_ind - 1)
    mask[0:i, :] = False
    j = max(col_ind - step_size, 0)
    mask[row_ind, 0:j] = False
    return mask

def update_box(row_ind, col_ind, box):
    box.set_xy((col_ind, row_ind))
    return box
    
# Load image and rot3 (for testing, using random data)
img = np.load('data/STEM img.npy')
rot3 = np.load('data/rot3 img.npy')

# Parameter settings
mask_size = rot3.shape[0]
step_size = 120

# Create a mask (initially masking everything)
mask = np.ones_like(rot3, dtype=bool)
masked_data = ma.array(rot3, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
ax1, ax2 = axes.ravel()

im1 = ax1.imshow(img, cmap='viridis')
im2 = ax2.imshow(masked_data, cmap='viridis', vmin=rot3.min(), vmax=rot3.max())

box = plt.Rectangle((0, 0), width=72, height=72, fill=False, ec='r')
ax2.add_patch(box)

# Compute positions for the sliding kernel
ijs = compute_ij(step_size, mask_size)

def update(frame):
    # here 'global box' is must
    global box
    i, j = ijs[frame]
    mask = update_mask(i, j, step_size, mask_size)
    masked_data.mask = mask  # Update mask directly
    im2.set_array(masked_data)
    box = update_box(i, j, box)
    return [im2, box]

# Create the animation
ani = FuncAnimation(
    fig,
    update,
    frames=len(ijs),
    interval=5,  # Interval between frames in milliseconds
    blit=True,  # Blit=False to avoid rendering issues
)